<a href="https://colab.research.google.com/github/DebbieJara/Adventure_works_analisis_financiero/blob/main/retail_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛍️ Retail Transactions — End-to-End Data Quality & Exploratory Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DebbieJara/retail-data-analysis/blob/main/retail_data_analysis.ipynb)

---

**Author:** Debbie Jara  
**Stack:** Python · pandas · NumPy · Matplotlib · Seaborn  
**Dataset:** Retail transaction records (e-commerce)

---

## 📌 Project Overview

This project applies a full data quality and exploratory analysis workflow to a retail transactions dataset. The goal was to assess data reliability, understand customer purchase behavior, and produce clean, analysis-ready data through a reusable and documented pipeline.

The dataset contains customer purchase records including order values, product categories, payment methods, and demographic information.

**Sections:**

1. Data Quality Audit
2. Missing Value Analysis
3. Reusable Cleaning Pipeline
4. Statistical Profiling
5. Distribution Visualizations
6. Outlier Detection
7. Feature Engineering

---

## ⚙️ Setup

The dataset `retail_transactions.csv` is included in this repository. No additional setup needed — just run all cells.

| Column | Type | Description |
|---|---|---|
| `order_id` | string | Unique order identifier |
| `order_date` | date | Purchase date |
| `customer_id` | int | Unique customer identifier |
| `product_category` | string | e.g. Fashion, Grocery, Sports |
| `price` | float | Unit price |
| `quantity` | float | Units purchased |
| `order_value` | float | Total transaction amount |
| `payment_method` | string | Payment method used |
| `city` | string | Customer city |
| `state` | string | Customer state |
| `customer_age` | float | Customer age |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

FILE_PATH = 'everpeak_retail.csv'
df = pd.read_csv(FILE_PATH)

print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

Dataset loaded: 5,008 rows x 11 columns


---
## 1. Data Quality Audit

The first step was to get a clear picture of what the data actually looks like before touching anything. I checked the overall structure, identified which columns had missing values, assessed cardinality to separate true categories from high-cardinality identifiers, and flagged any date records that fell outside the expected range.

In [3]:
import pandas as pd

FILE_PATH = 'everpeak_retail.csv'
df = pd.read_csv(FILE_PATH)

# 1.1 Structure overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5008 entries, 0 to 5007
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          5008 non-null   int64  
 1   order_date        5000 non-null   object 
 2   customer_id       5008 non-null   int64  
 3   product_category  5008 non-null   object 
 4   price             5008 non-null   int64  
 5   quantity          5008 non-null   int64  
 6   order_value       5008 non-null   int64  
 7   payment_method    5008 non-null   object 
 8   city              4908 non-null   object 
 9   state             4908 non-null   object 
 10  customer_age      4858 non-null   float64
dtypes: float64(1), int64(5), object(5)
memory usage: 430.5+ KB


In [4]:
# 1.2 Missing value counts for key segmentation columns
payment_missing = df['payment_method'].isna().sum()
city_missing    = df['city'].isna().sum()
state_missing   = df['state'].isna().sum()

print('payment_method missing:', payment_missing)
print('city missing:', city_missing)
print('state missing:', state_missing)

payment_method missing: 0
city missing: 100
state missing: 100


In [6]:
missingness = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().sum() / len(df)*100).round(2)
})
missingness = missingness[missingness['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missingness)

              missing_count  missing_pct
customer_age            150         3.00
city                    100         2.00
state                   100         2.00
order_date                8         0.16


In [ ]:
# 1.3 Cardinality check — distinguish IDs from categories
cols_to_check = ['customer_id', 'payment_method', 'city', 'state']

for col in cols_to_check:
    print(f'{col} — unique values: {df[col].nunique()}')

In [ ]:
# 1.4 Date validation — detect out-of-range years
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

invalid_year_count  = (df['order_date'].dt.year == 2026).sum()
missing_dates_count = df['order_date'].isna().sum()

print('Records with year 2026 (likely future-dated):', invalid_year_count)
print('Records with unparseable order_date:', missing_dates_count)

---
## 2. Missing Value Analysis

Rather than applying a single imputation strategy across the board, I first investigated the pattern behind each column's missing values. Specifically, I tested whether the null rate in `city` was consistent across payment method groups — if it varied, that would suggest the missingness was related to another variable in the dataset (MAR), which changes how it should be handled. I also compared the effect of dropping versus imputing on key numeric columns to make sure the chosen strategy didn't distort the distributions.

In [ ]:
# 2.1 Does missing city depend on payment_method?
missing_city_by_pay = df['city'].isna().groupby(df['payment_method']).mean()
print('Missing rate for city by payment method:')
print(missing_city_by_pay)

In [ ]:
# 2.2 Impact of imputation vs. dropping on order_value
before = df['order_value'].dropna().mean()

df['order_value_imputed'] = df['order_value'].fillna(df['order_value'].median())
after = df['order_value_imputed'].mean()

print(f'Mean (drop NaN):      {before:,.2f}')
print(f'Mean (impute median): {after:,.2f}')
print(f'Difference:           {abs(before - after):,.2f}')

In [ ]:
# 2.3 Comparing imputation strategies for customer_age
before = df['customer_age'].mean()

df['age_med']  = df['customer_age'].fillna(df['customer_age'].median())
df['age_mean'] = df['customer_age'].fillna(df['customer_age'].mean())

print(f'Original mean:           {before:.4f}')
print(f'After median imputation: {df["age_med"].mean():.4f}')
print(f'After mean imputation:   {df["age_mean"].mean():.4f}')
print()
print('Given the right skew in age distribution, median imputation was selected to avoid inflating the mean.')

---
## 3. Reusable Cleaning Pipeline

I structured the cleaning logic into small, single-purpose functions rather than writing it as a single script. This makes each step easier to test, audit, and reuse on similar datasets. The main pipeline function calls them in sequence, so the full cleaning process can be applied with one call.

In [ ]:
# 3.1 Helper functions

def replace_sentinels(df, sentinels, numeric_cols):
    """Replace known invalid sentinel values with NaN across multiple columns."""
    for col in numeric_cols:
        df[col] = df[col].replace(sentinels, pd.NA)
    return df


def fill_missing_numeric(df, cols_fill):
    """Convert columns to numeric and fill NaN with column mean."""
    for col in cols_fill:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col].fillna(df[col].mean(), inplace=True)
    return df


def strip_text_columns(df):
    """Remove leading/trailing whitespace from categorical columns."""
    text_cols = ['product_category', 'city', 'state']
    for col in text_cols:
        df[col] = df[col].str.strip()
    return df

In [ ]:
# 3.2 Main cleaning pipeline

def clean_retail_data(df):
    """Orchestrate all cleaning steps in the correct order."""
    invalid_sentinels = [-999, 999, 0, -1]
    numeric_cols      = ['customer_age', 'price']

    df = replace_sentinels(df, invalid_sentinels, numeric_cols)
    df = fill_missing_numeric(df, numeric_cols)
    df = strip_text_columns(df)
    return df


print('Before cleaning:')
print(df[['customer_age', 'price']].isna().sum())

df = clean_retail_data(df)

print('\nAfter cleaning:')
print(df[['customer_age', 'price']].isna().sum())

---
## 4. Statistical Profiling

With clean data in hand, I generated descriptive statistics broken down by product category, and compared mean vs. median for key financial columns. When mean and median diverge significantly, it's usually a signal of skew or extreme values — which informs both visualization choices and whether parametric methods are appropriate downstream.

In [ ]:
# 4.1 Numeric summary by product category
numeric_cols = ['order_value', 'customer_age', 'price', 'quantity']

for category in ['Fashion', 'Sports']:
    subset = df[df['product_category'] == category]
    print(f'\n--- {category} ---')
    print(subset[numeric_cols].describe().round(2))

In [ ]:
# 4.2 Mean vs. median — detecting skew in order_value
mean_val   = df['order_value'].mean()
median_val = df['order_value'].median()

print(f'Mean:   {mean_val:,.2f}')
print(f'Median: {median_val:,.2f}')

if abs(mean_val - median_val) / median_val > 0.10:
    print('\nMean and median diverge by more than 10% — distribution is likely right-skewed with outliers.')
else:
    print('\nMean and median are close — distribution is approximately symmetric.')

In [ ]:
# 4.3 Categorical frequency distribution
cat_cols = ['product_category', 'payment_method', 'city', 'state']

for col in cat_cols:
    print(f'\n--- {col} ---')
    print('Absolute frequency:')
    print(df[col].value_counts())
    print('Relative frequency:')
    print(df[col].value_counts(normalize=True).round(3))

---
## 5. Distribution Visualizations

I plotted histograms for the main numeric variables to assess their shape, and used side-by-side boxplots to compare order value spread across product categories. Boxplots are particularly useful here because they surface the median, interquartile range, and potential outliers in a single view — which connects directly to the outlier detection in the next section.

In [ ]:
# 5.1 Price distribution histogram
fig, ax = plt.subplots(figsize=(10, 4))
counts, bin_edges, _ = ax.hist(
    df['price'].dropna(), bins=10, range=(0, 1000),
    color='steelblue', edgecolor='white', alpha=0.85
)
ax.set_xticks(bin_edges.round(0))
ax.set_xlabel('Unit Price', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Price Distribution (capped at $1,000)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Customer age distribution histogram
fig, ax = plt.subplots(figsize=(10, 4))
counts, bin_edges, _ = ax.hist(
    df['customer_age'].dropna(), bins=10,
    color='coral', edgecolor='white', alpha=0.85
)
ax.set_xticks(bin_edges.round(0))
ax.set_xlabel('Customer Age', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Customer Age Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Boxplots — order_value by category
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, category in zip(axes, ['Fashion', 'Sports']):
    subset = df[df['product_category'] == category]['order_value']
    sns.boxplot(x=subset, color='skyblue', ax=ax)
    ax.set_xlabel('Order Value', fontsize=11)
    ax.set_title(f'Order Value — {category}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 6. Outlier Detection

I applied both the IQR method and Z-score to `order_value` and `price`, then compared how many records each method flagged. The IQR method tends to be more aggressive with right-skewed data, while Z-score is stricter and better suited to near-normal distributions. Running both gives a clearer picture of which extreme values are genuine anomalies versus expected high-value transactions.

In [ ]:
# 6.1 IQR method on order_value
Q1  = df['order_value'].quantile(0.25)
Q3  = df['order_value'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iqr = df[(df['order_value'] < lower) | (df['order_value'] > upper)]

print(f'Q1: {Q1:,.0f}  |  Q3: {Q3:,.0f}  |  IQR: {IQR:,.0f}')
print(f'Lower fence: {lower:,.0f}  |  Upper fence: {upper:,.0f}')
print(f'\nOutliers detected (IQR): {len(outliers_iqr):,} records')

In [ ]:
# 6.2 Z-score method on order_value
mean_ov = df['order_value'].mean()
std_ov  = df['order_value'].std()

df['z_order_value'] = (df['order_value'] - mean_ov) / std_ov

outliers_z = df[df['z_order_value'].abs() > 3]
print(f'Outliers detected (|Z| > 3): {len(outliers_z):,} records')

In [ ]:
# 6.3 Method comparison on price
Q1_p  = df['price'].quantile(0.25)
Q3_p  = df['price'].quantile(0.75)
IQR_p = Q3_p - Q1_p
lower_p = Q1_p - 1.5 * IQR_p
upper_p = Q3_p + 1.5 * IQR_p

mean_p = df['price'].mean()
std_p  = df['price'].std()
df['z_price'] = (df['price'] - mean_p) / std_p

iqr_count = len(df[(df['price'] < lower_p) | (df['price'] > upper_p)])
z_count   = len(df[df['z_price'].abs() > 3])

print(f'Price outliers — IQR method:     {iqr_count:,}')
print(f'Price outliers — Z-score method: {z_count:,}')
print('\nIQR flagged more records, consistent with a right-skewed price distribution.')

---
## 7. Feature Engineering — Customer Segmentation

To add analytical value beyond raw columns, I created three new segmentation features. The first uses `np.where()` for a simple binary split on purchase volume. The second combines volume and age into four customer profiles using `apply()` with a custom function. The third segments by payment method and volume to identify behavioral patterns that could be relevant for targeted strategies.

In [ ]:
# 7.1 Simple binary segmentation with np.where()
df['volume_segment'] = np.where(
    df['quantity'] > 22, 'High Volume', 'Low Volume'
)

print('Volume segment distribution:')
print(df['volume_segment'].value_counts())

In [ ]:
# 7.2 Multi-condition segmentation with apply()

def classify_customer(row):
    """Classify customers by purchase volume and age group."""
    age = row['customer_age']
    qty = row['quantity']

    if pd.isna(age) or pd.isna(qty):
        return 'Data Error'

    if qty > 22:
        return 'Sr. High Volume' if age > 55 else 'Jr. High Volume'
    else:
        return 'Sr. Low Volume' if age > 55 else 'Jr. Low Volume'


df['customer_segment'] = df.apply(classify_customer, axis=1)

print('Customer segment breakdown:')
print(df['customer_segment'].value_counts())

In [ ]:
# 7.3 Payment behaviour segmentation

def classify_payment(row):
    """Segment transactions by payment type and purchase volume."""
    method = row['payment_method']
    qty    = row['quantity']

    if pd.isna(method) or pd.isna(qty):
        return 'Data Error'

    is_card = method in ('credit_card', 'debit_card')
    prefix  = 'card' if is_card else 'no_card'
    volume  = 'high' if qty > 22 else 'low'
    return f'{prefix}_{volume}_volume'


df['payment_segment'] = df.apply(classify_payment, axis=1)

print('Payment segment breakdown:')
print(df['payment_segment'].value_counts())

---
## 8. Summary & Key Findings

| Topic | Finding |
|---|---|
| Missing values | `city` and `state` had the highest null rates; `order_date` contained future-dated records requiring filtering |
| Missingness pattern | Null rate for `city` varied across payment method groups, indicating MAR — handled with conditional logic rather than global imputation |
| Distribution shape | `order_value` and `price` are right-skewed; median was used as the preferred central tendency measure |
| Outlier detection | IQR flagged more records than Z-score on skewed columns, consistent with expected behavior — extreme values were isolated for review rather than dropped |
| Customer segments | Four segments created from age and purchase volume; payment-based segmentation adds a behavioral dimension for downstream targeting |

---

All transformations are encapsulated in reusable functions. The pipeline can be applied to any dataset sharing this schema by updating `FILE_PATH`.